# Text Classification

Oskar Jonsson
2000211138375

LLM Usage:

In [ ]:
%pip install -q pandas matplotlib numpy torch
%pip install -q transformers[sentencepiece] accelerate
%pip install -q scikit-learn
%pip install -q peft datasets
%pip install -U transformers
%pip install -U accelerate

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
import torch
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
PyTorch version: 2.6.0+cu124
CUDA available: True


In [2]:
class SentimentClassifier:
    def __init__(self, model_name="microsoft/deberta-v3-small", num_labels=2):
        """
        Initialize sentiment classifier with a pre-trained model and tokenizer.
        
        :param model_name: The name of the pre-trained model to use.
        :param num_labels: The number of labels for classification.
        """
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        
        self.apply_lora()
        
        self.model.print_trainable_parameters()
        
        self.model = self.model.to(self.device) 

        
    def apply_lora(self):
        """
        Apply LoRA (Low-Rank Adaptation) to the model for fine-tuning.
        """
        lora_config = LoraConfig(
            r=16,  
            lora_alpha=32,  
            target_modules=["query_proj", "value_proj", "key_proj", "output.dense"],  
            lora_dropout=0.1,
            bias="none",
            task_type="SEQ_CLS"
        )
        
        self.model = get_peft_model(self.model, lora_config)
        
    def load_data(self, path):
        """
        Load and preprocess dataset from txt file.
        
        :param path: The path to the dataset file.
        """
        
        texts = []
        labels = []

        with open(path, "r", encoding="utf8") as f:
            for line in f:

                label, text = line.strip().split("\t", 1)

                labels.append(int(label))
                texts.append(text)

        return pd.DataFrame({"text": texts, "label": labels})
    
    def tokenize(self, examples):
        """
        Tokenize the input texts using the tokenizer.
        
        :param examples: A batch of examples to tokenize.
        """
        
        return self.tokenizer(examples["text"], truncation=True, padding=False, max_length=512)
    
    def prepare_datasets(self, train_path, val_path, test_path):
        """
        Prepare datasets for training, validation, and testing.
        
        :param train_path: The path to the training dataset file.
        :param val_path: The path to the validation dataset file.
        :param test_path: The path to the testing dataset file.
        """
        
        train_df = self.load_data(train_path)
        val_df = self.load_data(val_path)
        test_df = self.load_data(test_path)
        
        print("Class distribution in training:")
        print(train_df['label'].value_counts())
        print("\nClass distribution in validation:")
        print(val_df['label'].value_counts())
    
        self.test_labels = test_df["label"]
        
        train_dataset = Dataset.from_pandas(train_df)
        val_dataset = Dataset.from_pandas(val_df)
        test_dataset = Dataset.from_pandas(test_df)

        train_dataset = train_dataset.map(self.tokenize, batched=True)
        val_dataset = val_dataset.map(self.tokenize, batched=True)
        test_dataset = test_dataset.map(self.tokenize, batched=True)
        
        train_dataset = train_dataset.rename_column("label", "labels")
        val_dataset = val_dataset.rename_column("label", "labels")
        test_dataset = test_dataset.rename_column("label", "labels")
        
        train_dataset.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
        val_dataset.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
        test_dataset.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

        return train_dataset, val_dataset, test_dataset
    
    def compute_metrics(self, eval_pred):
        """
        Compute accuracy metric for evaluation.
        
        :param eval_pred: The predictions and labels from the evaluation.
        """

        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)

        acc = accuracy_score(labels, preds)

        return {"accuracy": acc}
    
    
    def train(self, train_dataset, val_dataset):
        """
        Train the model using the Trainer API.
        
        :param train_dataset: The training dataset.
        :param val_dataset: The validation dataset.
        """

        
        
        training_args = TrainingArguments(
            output_dir="./results",
            learning_rate=2e-4,
            per_device_train_batch_size=8,     
            per_device_eval_batch_size=8,
            num_train_epochs=3,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=50,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            warmup_steps=100,
            optim="adamw_torch",
            gradient_accumulation_steps=4,    
            weight_decay=0.01,
            fp16=False,                       
            bf16=False,                       
            max_grad_norm=1.0,                
            report_to="none",
        )


        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            processing_class=self.tokenizer,
            compute_metrics=self.compute_metrics,
            data_collator=DataCollatorWithPadding(self.tokenizer)
        )


        self.trainer.train()
        
    def evaluate(self, test_dataset):
        """
        Evaluate the model on the test dataset and print accuracy and classification report.
        
        :param test_dataset: The testing dataset.
        """

        preds = self.trainer.predict(test_dataset)

        y_pred = np.argmax(preds.predictions, axis=1)

        print("Accuracy:", accuracy_score(self.test_labels, y_pred))
        print(classification_report(self.test_labels, y_pred))


        

In [ ]:
classifier = SentimentClassifier()

train_dataset, val_dataset, test_dataset = classifier.prepare_datasets(
    "ReviewBaseTraining.txt",
    "ReviewBaseValidation.txt",
    "ReviewBaseTest.txt"
)

classifier.train(train_dataset, val_dataset)

classifier.evaluate(test_dataset)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight       

trainable params: 960,002 || all params: 142,856,452 || trainable%: 0.6720
Class distribution in training:
label
0    5000
1    5000
Name: count, dtype: int64

Class distribution in validation:
label
0    2500
1    2500
Name: count, dtype: int64


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]